# 4. AIND Ephys Postprocessing

Build an `AINDEPhysPostprocessingScanConfig`, expand it with `GridScanGenerationTask`, and run the `AINDEPhysPostprocessingTask` for each coordinate. The task itself clones [aind-ephys-postprocessing](https://github.com/AllenNeuralDynamics/aind-ephys-postprocessing) on first use, patches `run_capsule.py` for spikeinterface API drift (`qm_params` → `metric_params`), seeds its `data/` with the preprocessing + spike-sorting outputs, writes a `params_obi.json`, runs `code/run_capsule.py`, and copies the resulting `postprocessed_<name>.zarr` (a `SortingAnalyzer`) into the single config's `coordinate_output_root`.

Some default per-metric parameters assume minute-long recordings (e.g. `presence_ratio.bin_duration_s=60`); we override those for the 8-second toy via the `quality_metrics` block. The default metric list also has `l_ratio` and `isolation_distance` replaced by `mahalanobis` (the way newer spikeinterface re-organised them); the `template_metrics.sparsity` argument is dropped since newer `ComputeTemplateMetrics` rejects it.

## Imports and prerequisites

In [1]:
import shutil
import subprocess
import sys
from pathlib import Path

import obi_one as obi
from obi_one.scientific.tasks.aind_ephys._04_postprocessing.blocks import QualityMetrics

In [2]:
subprocess.run(
    [
        "uv", "pip", "install", "--python", sys.executable,
        "aind-data-schema", "scikit-learn", "numba",
    ],
    check=True,
)

Using Python 3.12.9 environment at: /Users/james/Documents/obi/code/obi-main/obi-one/.venv


Resolved 21 packages in 485ms
Uninstalled 2 packages in 15ms
Installed 2 packages in 5ms
 - pydantic==2.12.5
 + pydantic==2.11.10
 - pydantic-core==2.41.5
 + pydantic-core==2.33.2


CompletedProcess(args=['uv', 'pip', 'install', '--python', '/Users/james/Documents/obi/code/obi-main/obi-one/.venv/bin/python', 'aind-data-schema', 'scikit-learn', 'numba'], returncode=0)

## Build the scan config

We point `preprocessing_output_path` at the 8-second preprocessing run done inside `03_aind_ephys_spikesort_kilosort4.ipynb` (`/tmp/aind-ephys-preprocessing/results/`) and `spikesort_output_path` at this folder's `output/03_spikesort_results/`. The `quality_metrics` block lowers the bin sizes that assume minute-long recordings.

In [3]:
preprocessing_output_path = Path("/tmp/aind-ephys-preprocessing/results").resolve()
assert preprocessing_output_path.exists(), (
    f"{preprocessing_output_path} not found. Run 03_aind_ephys_spikesort_kilosort4.ipynb first."
)

spikesort_output_path = (Path.cwd() / "output/03_spikesort_results").resolve()
assert spikesort_output_path.exists(), (
    f"{spikesort_output_path} not found. Run 03_aind_ephys_spikesort_kilosort4.ipynb first."
)

scan_config = obi.AINDEPhysPostprocessingScanConfig(
    initialize=obi.AINDEPhysPostprocessingScanConfig.Initialize(
        preprocessing_output_path=preprocessing_output_path,
        spikesort_output_path=spikesort_output_path,
        n_jobs=1,
    ),
    quality_metrics=QualityMetrics(
        presence_ratio_bin_duration_s=1.0,
        firing_range_bin_size_s=1.0,
        amplitude_cv_min_num_bins=1,
        amplitude_cv_average_num_spikes_per_bin=5,
    ),
)

## Generate the grid scan and run the postprocessing task

In [4]:
grid_scan = obi.GridScanGenerationTask(
    form=scan_config,
    output_root="../../../obi-output/04_aind_ephys_postprocessing/grid_scan",
)
grid_scan.execute()

obi.run_tasks_for_generated_scan(grid_scan)

[2026-04-29 16:43:32,207] INFO: Cloning https://github.com/AllenNeuralDynamics/aind-ephys-postprocessing.git -> /tmp/aind-ephys-postprocessing


Cloning into '/tmp/aind-ephys-postprocessing'...


[2026-04-29 16:43:33,007] INFO: Patched run_capsule.py: qm_params -> metric_params


[2026-04-29 16:43:33,079] INFO: Seeded 1 preprocessed + 1 spikesorted recording(s) into /tmp/aind-ephys-postprocessing/data


[2026-04-29 16:43:33,079] INFO: Running python -u run_capsule.py --params params_obi.json --n-jobs 1


[2026-04-29 16:43:59,252] INFO: Running postprocessing with the following parameters:
	USE_MOTION_CORRECTED: False
	N_JOBS: -1

POSTPROCESSING
Found 0 json configurations
	Processing block0_None_recording1
	Loaded binary recording from JSON
	Loading lazy recording from JSON
	Creating sorting analyzer
NumExpr defaulting to 14 threads.
	Number of original units: 14 -- Number of units after de-duplication: 10
	Setting temporary binary recording
	Computing all postprocessing extensions
	Computing quality metrics
	Saving SortingAnalyzer to zarr
POSTPROCESSING time: 25.03s



[PosixPath('../../../obi-output/04_aind_ephys_postprocessing/grid_scan/AINDEPhysPostprocessingSingleConfig')]

## Copy the per-coordinate postprocessing results next to the notebook

In [5]:
local_results_dir = Path.cwd() / "output/04_postprocessing_results"
if local_results_dir.exists():
    shutil.rmtree(local_results_dir)
local_results_dir.mkdir(parents=True, exist_ok=True)

for single_config in grid_scan.single_configs:
    coord_dir = Path(single_config.coordinate_output_root)
    dest = local_results_dir / str(single_config.idx)
    dest.mkdir(parents=True, exist_ok=True)
    for entry in coord_dir.iterdir():
        target = dest / entry.name
        if target.exists():
            shutil.rmtree(target) if target.is_dir() else target.unlink()
        if entry.is_dir():
            shutil.copytree(entry, target)
        else:
            shutil.copy2(entry, target)

# Mirror the first coordinate's outputs at the top level so downstream
# notebooks can read a stable `output/04_postprocessing_results/...` path.
first = local_results_dir / "0"
if first.exists():
    for entry in first.iterdir():
        target = local_results_dir / entry.name
        if target.exists():
            shutil.rmtree(target) if target.is_dir() else target.unlink()
        if entry.is_dir():
            shutil.copytree(entry, target)
        else:
            shutil.copy2(entry, target)

for p in sorted(local_results_dir.iterdir()):
    print(" ", p.name)

  0
  data_process_postprocessing_block0_None_recording1.json
  obi_one_coordinate.json
  postprocessed_block0_None_recording1.zarr


## Load the SortingAnalyzer and inspect quality metrics

In [6]:
import spikeinterface.full as si

zarr_dirs = [
    p
    for p in local_results_dir.iterdir()
    if p.is_dir() and p.name.startswith("postprocessed_") and p.suffix == ".zarr"
]
print("Postprocessed analyzers:", [p.name for p in zarr_dirs])

if zarr_dirs:
    analyzer = si.load_sorting_analyzer(zarr_dirs[0])
    print(analyzer)
    print("Extensions computed:", sorted(analyzer.get_loaded_extension_names()))

    qm_ext = analyzer.get_extension("quality_metrics")
    if qm_ext is not None:
        qm = qm_ext.get_data()
        print()
        print(qm)

Postprocessed analyzers: ['postprocessed_block0_None_recording1.zarr']


SortingAnalyzer: 70 channels - 10 units - 1 segments - zarr - sparse
Loaded 13 extensions: correlograms, isi_histograms, noise_levels, principal_components, quality_metrics, random_spikes, spike_amplitudes, spike_locations, template_metrics, template_similarity, templates, unit_locations, waveforms
Extensions computed: ['correlograms', 'isi_histograms', 'noise_levels', 'principal_components', 'quality_metrics', 'random_spikes', 'spike_amplitudes', 'spike_locations', 'template_metrics', 'template_similarity', 'templates', 'unit_locations', 'waveforms']

    amplitude_cutoff  amplitude_cv_median  amplitude_cv_range  \
0                NaN             0.103542            0.142882   
2                NaN             0.040664            0.077025   
3                NaN             0.575957            0.000000   
5                NaN             0.628303            0.359084   
8                NaN             0.267783            0.303546   
9                NaN             0.165905          